In [3]:
import folium, json, pickle, copy
from IPython.display import display
import requests
from DataFetcher import DataFetcher

In [ ]:
things_data = list()
things_urls = "https://tld.iot.hamburg.de/v1.1/Things?$top=1000"


response = requests.get(things_urls)
fetched_data = response.json().get("value", [])
things_data.extend(fetched_data)
while True:
    next_link = response.json().get("@iot.nextLink")
    if not next_link:  # break if last page is reached
        break
    # requesting next 1000 Things
    response = requests.get(next_link)
    if response.status_code != 200:  # exit data fetching if request failed
        print(f"Failed to fetch data. Status code: {response.status_code}")
        break
    # Fetching data
    fetched_data = response.json().get("value", [])
    things_data.extend(fetched_data)

In [ ]:
thing_locations = dict()

things_no_locations = list()

for thing in things_data:
    # print(thing["@iot.id"])
    locations_url = thing["Locations@iot.navigationLink"]
    locations_data = requests.get(locations_url).json()

    # print(locations_data)

    if "value" in locations_data and len(locations_data["value"]) > 0:
        # In case of key
        if "geometry" not in locations_data["value"][0]["location"].keys():
            print(f"No location data available for Thing ID: {thing['@iot.id']}")
            things_no_locations.append(thing["@iot.id"])
            continue

        coordinates = locations_data["value"][0]["location"]["geometry"]["coordinates"]
        # If found, collect the coordinates
        if coordinates:
            thing_locations[thing["@iot.id"]] = coordinates
    else:
        print(f"No location data available for Thing ID: {thing['@iot.id']}")
        things_no_locations.append(thing["@iot.id"])

    thing_locations[thing["@iot.id"]] = coordinates

In [ ]:
thing_locations_copy = copy.deepcopy(thing_locations)
thing_nolocations_copy = copy.deepcopy(things_no_locations)

with open("things_locationsx.pkl", "wb") as file:
    pickle.dump(thing_locations_copy, file)

with open("things_no_locationsx.pkl", "wb") as file:
    pickle.dump(thing_nolocations_copy, file)

print("Saved successfully!")

In [ ]:
test = dict()

with open("data/things_locations.pkl", "rb") as file:
    test = pickle.load(file)

test[18670]